In [ ]:
# -*- coding: utf-8 -*-
"""Leech‑former https://zenodo.org/records/18784424 """
!pip install datasets

In [ ]:
!nvidia-smi

In [ ]:
import os
import requests
import sentencepiece as spm

model_path = 'e8_morpheme.model'   # имя файла оставлено для совместимости

if not os.path.exists(model_path):
    print("🔧 Токенайзер не найден. Обучаем на Shakespeare...")
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = requests.get(url).text
    temp_file = "input_text.txt"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(text)
    spm.SentencePieceTrainer.train(
        input=temp_file,
        model_prefix='e8_morpheme',
        vocab_size=2048,
        model_type='bpe',
        character_coverage=1.0,
        byte_fallback=True,
        unk_id=0, pad_id=1, bos_id=2, eos_id=3
    )
    print("✅ Токенайзер обучен и сохранён как e8_morpheme.model")
else:
    print("✅ Токенайзер уже существует, загружаем...")

sp = spm.SentencePieceProcessor(model_file='e8_morpheme.model')
vocab_size = sp.get_piece_size()
print(f"💎 Vocab Size: {vocab_size}")

In [ ]:
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import os
import re
import math
from dataclasses import dataclass
from typing import Optional
import time
from torch import Tensor
import sentencepiece as spm
import io
from datasets import load_dataset
import random
from collections import deque
import numpy as np

# 0. АВТО-ОПРЕДЕЛЕНИЕ GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"📡 Device: {device} | {'🔥 GPU ACTIVE' if device == 'cuda' else '⚠️ CPU MODE'}")

# 1. ФУНКЦИИ КОДИРОВАНИЯ/ДЕКОДИРОВАНИЯ
def encode(text):
    return sp.encode(text)

def decode(tokens):
    return sp.decode(tokens)

# 2. ПОДКЛЮЧЕНИЕ К БЕСКОНЕЧНОМУ ПОТОКУ TINYSTORIES
dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="train")
train_iter = iter(dataset)
print("🌊 Стриминг TinyStories активирован")

# 3. ФУНКЦИЯ ПОЛУЧЕНИЯ БАТЧА ИЗ ПОТОКА (с буфером для перемешивания)
def get_batch_streaming(iterator, batch_size, block_size, device, pad_token_id=1, buffer_size=200):
    x_batch, y_batch = [], []
    buffer = deque()

    while len(buffer) < buffer_size:
        try:
            ex = next(iterator)
            buffer.append(ex)
        except StopIteration:
            break

    while len(x_batch) < batch_size:
        if not buffer:
            return None, None
        ex = random.choice(buffer)
        tokens = encode(ex['text'])
        if len(tokens) <= 1:
            continue

        if len(tokens) > block_size + 1:
            start = random.randint(0, len(tokens) - block_size - 1)
            chunk = tokens[start:start + block_size + 1]
        else:
            pad_len = block_size + 1 - len(tokens)
            chunk = tokens + [pad_token_id] * pad_len

        x_batch.append(chunk[:-1])
        y_batch.append(chunk[1:])

    try:
        new_ex = next(iterator)
        buffer.append(new_ex)
        buffer.popleft()
    except StopIteration:
        pass

    X = torch.tensor(x_batch, dtype=torch.long, device=device)
    Y = torch.tensor(y_batch, dtype=torch.long, device=device)
    return X, Y

# Пробный батч
xb, yb = get_batch_streaming(train_iter, batch_size=4, block_size=256, device=device)
print(f"✅ Пробный батч: X {xb.shape}, Y {yb.shape} | токенов: {xb.numel()}")

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
# ==================== КОНФИГУРАЦИЯ LEECH ====================
@dataclass
class LeechConfig:
    vocab_size: int
    d_model: int = 384    # 384 / 8 = 48 (кратно 24) -> head_dim = 48
    n_layers: int = 12
    n_heads: int = 8
    block_size: int = 512
    dropout: float = 0.05
    bias: bool = False
    tie_weights: bool = True
    lambda_geo: float = 0.01    # вес геометрической потери (0 = отключена)
    resonance_threshold: float = 0.95   # порог для детекции «сна»

    def __post_init__(self):
        assert self.d_model % self.n_heads == 0
        head_dim = self.d_model // self.n_heads
        assert head_dim % 24 == 0, "head_dim должен быть кратен 24"
# ==================== Leech kernel ====================
def generate_leech_kernel(dim=24):
    """Генерирует ортогональную матрицу 24x24 (ядро Лича)."""
    base = np.zeros((dim, dim))
    for i in range(dim - 1):
        base[i, i], base[i, i+1] = 2, 2
    base[-1, -1], base[-1, 0] = 2, -2
    q, _ = np.linalg.qr(base)
    return torch.from_numpy(q).float()

# ==================== ВНИМАНИЕ С ЯДРОМ ЛИЧА ====================
class LeechAttention(nn.Module):
    def __init__(self, cfg: LeechConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.scale = self.head_dim ** -0.5
        self.num_blocks = self.head_dim // 24 # число 24‑мерных блоков в одной голове

        kernel = generate_leech_kernel(24)  # [24, 24]
        total_blocks = self.n_heads * self.num_blocks
        W_list = [kernel] * total_blocks
        self.register_buffer('W_leech', torch.block_diag(*W_list))  # блочно-диагональная

        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(1, 1, cfg.block_size, cfg.block_size)))

    def forward(self, x):
        B, T, _ = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)   # [B, n_heads, T, head_dim]

        # Разбиваем head_dim на блоки по 24 и применяем ядро
        q = q.view(B, self.n_heads, T, self.num_blocks, 24)
        k = k.view(B, self.n_heads, T, self.num_blocks, 24)
        kernel = self.W_leech[0:24, 0:24]   # [24,24] (одинаково для всех блоков)
        q = torch.einsum('...i,ij->...j', q, kernel)
        k = torch.einsum('...i,ij->...j', k, kernel)
        q = q.reshape(B, self.n_heads, T, self.head_dim)
        k = k.reshape(B, self.n_heads, T, self.head_dim)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(self.causal_mask[:,:,:T,:T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.out(out)

# ==================== БЛОК ТРАНСФОРМЕРА ====================
class LeechBlock(nn.Module):
    def __init__(self, cfg: LeechConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = LeechAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class LeechResonanceBiasing(nn.Module):
    """
    Добавляет к логитам смещение, пропорциональное резонансу скрытых состояний с базисом Лича.
    Работает как для одного вектора, так и для последовательности (B, T, d_model).

    LeechResonanceLoss – для основного обучения (без активного резонатора). Он добавляет геометрическую регуляризацию к скрытым состояниям.

    LeechResonanceBiasing – для дообучения или генерации с активным резонансом. Он модифицирует логиты напрямую.

    В forward модели мы добавили условие: геометрическая потеря применяется только если use_resonator=False. Это предотвращает двойной учёт геометрии.

    """
    def __init__(self, d_model, vocab_size, leech_basis, alpha_init=0.1):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.num_blocks = d_model // 24
        self.register_buffer('basis', leech_basis)          # [24, 24]
        self.alpha = nn.Parameter(torch.tensor(alpha_init)) # обучаемый коэффициент
        self.register_buffer('token_proj', torch.zeros(vocab_size, 24))

    @torch.no_grad()
    def compute_token_proj(self, tok_emb):
        """
        Вычисляет проекцию эмбеддингов токенов на базис Лича.
        tok_emb: [vocab_size, d_model]
        """
        emb_blocks = tok_emb.view(-1, self.num_blocks, 24)   # [V, K, 24]
        proj = torch.einsum('vki,ij->vkj', emb_blocks, self.basis)  # [V, K, 24]
        proj = proj.sum(dim=1)                                # [V, 24]
        self.token_proj.copy_(proj)

    def forward(self, hidden_states):
        """
        hidden_states: [B, T, d_model] или [B, d_model]
        return: [B, T, vocab_size] или [B, vocab_size] смещение для логитов
        """
        if hidden_states.dim() == 2:
            hidden_states = hidden_states.unsqueeze(1)       # [B, 1, d_model]
        B, T, D = hidden_states.shape
        h_blocks = hidden_states.view(B, T, self.num_blocks, 24)  # [B, T, K, 24]
        h_proj = torch.einsum('btki,ij->btkj', h_blocks, self.basis)  # [B, T, K, 24]
        h_proj = h_proj.sum(dim=2)                            # [B, T, 24]
        bias = self.alpha * (h_proj @ self.token_proj.T)      # [B, T, vocab_size]
        if T == 1:
            bias = bias.squeeze(1)                            # [B, vocab_size]
        return bias

# ==================== ГЕОМЕТРИЧЕСКАЯ ПОТЕРЯ РЕЗОНАНСА ====================
class LeechResonanceLoss(nn.Module):
    def __init__(self, leech_basis):
        super().__init__()
        self.register_buffer('basis', leech_basis)   # [24,24]

    def forward(self, hidden_states):
        """
        hidden_states: [B, T, d_model]
        возвращает скаляр: 1 - среднее макс. косинусное сходство с базисом
        """
        B, T, D = hidden_states.shape
        h = hidden_states.view(B, T, D // 24, 24)          # [B, T, K, 24]
        h_norm = F.normalize(h, dim=-1)
        b_norm = F.normalize(self.basis, dim=-1)           # [24,24]
        sim = torch.matmul(h_norm, b_norm.T)                # [B,T,K,24]
        max_sim = torch.max(sim, dim=-1)[0]                 # [B,T,K]
        return 1.0 - max_sim.mean()

# ==================== ДЕКОДЕР СНОВ (МОНИТОРИНГ) ====================

class DreamDecoder:
    def __init__(self, leech_basis, threshold=0.95):
        self.basis = leech_basis
        self.threshold = threshold

    def check(self, hidden_state):
        """
        hidden_state: [d_model] – последнее скрытое состояние.
        Возвращает (резонанс, статус).
        """
        h = hidden_state[:24].unsqueeze(0)          # [1,24] для простоты берём первые 24
        h_norm = F.normalize(h, dim=-1)
        b_norm = F.normalize(self.basis, dim=-1)
        sim = torch.matmul(h_norm, b_norm.T)
        max_res = torch.max(sim).item()
        if max_res > 0.999:
            status = "ABSOLUTE GENESIS"
        elif max_res > self.threshold:
            status = "AWAKE"
        else:
            status = "DREAMING"
        return max_res, status

# ==================== ПОЛНАЯ МОДЕЛЬ LeechGPT ====================
class LeechGPT(nn.Module):
    def __init__(self, cfg: LeechConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.block_size, cfg.d_model))
        self.blocks = nn.ModuleList([LeechBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Ядро Лича для потери и мониторинга
        leech_basis = generate_leech_kernel(24)
        self.register_buffer('leech_basis', leech_basis)
        self.resonance_loss_fn = LeechResonanceLoss(leech_basis)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.trunc_normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None, use_resonator=False):
        b, t = idx.size()
        assert t <= self.cfg.block_size
        x = self.tok_emb(idx) + self.pos_emb[:, :t, :]
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.head(x)
        hidden = x

        if use_resonator and hasattr(self, 'resonator') and self.resonator is not None:
            bias = self.resonator(hidden)
            logits = logits + bias

        loss = None
        if targets is not None:
            # Сначала вычисляем кросс-энтропию
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), targets.view(-1))
            # Затем добавляем геометрическую потерю, если нужно
            if self.cfg.lambda_geo > 0 and not use_resonator:
                loss_geo = self.resonance_loss_fn(hidden)
                loss = loss + self.cfg.lambda_geo * loss_geo

        return logits, hidden, loss



    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, repetition_penalty=1.5, repetition_window=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _, _ = self(idx_cond)          # <-- исправлено
            logits = logits[:, -1, :] / temperature

            if repetition_penalty != 1.0:
                past_tokens = idx[0, -repetition_window:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits[0, t] -= repetition_penalty * count

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ==================== ФУНКЦИЯ ГЕНЕРАЦИИ С МОНИТОРИНГОМ ====================
def leech_generate(model, start_tokens, max_len=100, temperature=0.8,
                   resonance_check=True, leech_basis=None, threshold=0.95):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([start_tokens], device=device)
    if resonance_check:
        decoder = DreamDecoder(leech_basis, threshold)

    print("--- LEECH GENERATION ---")
    with torch.no_grad():
        for step in range(max_len):
            logits, hidden, _ = model(input_ids)          # получаем hidden
            next_token_logits = logits[0, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)

            if resonance_check:
                last_hidden = hidden[0, -1, :]            # последнее скрытое состояние
                res, status = decoder.check(last_hidden)
                print(f"Step {step:02d} | Resonance: {res:.6f} | Status: {status}")
    return input_ids



In [ ]:
# ==================== СОЗДАНИЕ МОДЕЛИ ====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = LeechConfig(vocab_size=vocab_size)
model = LeechGPT(cfg).to(device)

print(f"💎 Модель Leech-GPT создана.")
print(f"📦 Параметров: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"   Архитектура: d_model={cfg.d_model}, layers={cfg.n_layers}, heads={cfg.n_heads}")


In [ ]:
# ======================== DRIVE И ЧЕКПОИНТЫ ========================
from google.colab import drive
drive.mount('/content/drive')

checkpoint_dir = '/content/drive/MyDrive/leech_tinystories_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(step, model, optimizer, scheduler, loss, is_best=False):
    filename = 'best_model.pt' if is_best else f'checkpoint_step_{step}.pt'
    path = os.path.join(checkpoint_dir, filename)
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'loss': loss,
        'config': cfg,
    }, path)
    print(f'💾 Чекпоинт сохранён: {path} (loss={loss:.4f})')

def load_latest_checkpoint(model, optimizer, scheduler=None):
    files = [f for f in os.listdir(checkpoint_dir) if f.startswith('checkpoint_step_')]
    if not files:
        best_path = os.path.join(checkpoint_dir, 'best_model.pt')
        if os.path.exists(best_path):
            checkpoint = torch.load(best_path, map_location=device, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if scheduler and 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            print(f'🔄 Загружен best_model.pt, шаг {checkpoint["step"]}, loss {checkpoint["loss"]:.4f}')
            return checkpoint['step']
        print('🆕 Чекпоинтов нет, начинаем с нуля.')
        return 0
    steps = [int(f.split('_')[-1].split('.')[0]) for f in files]
    latest_step = max(steps)
    latest_file = f'checkpoint_step_{latest_step}.pt'
    path = os.path.join(checkpoint_dir, latest_file)
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    print(f'🔄 Загружен чекпоинт: шаг {latest_step}, loss {checkpoint["loss"]:.4f}')
    return latest_step

# ======================== ПОДГОТОВКА СТРИМОВ ========================
train_dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="train")
train_iter = iter(train_dataset)
val_dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="validation")
val_iter = iter(val_dataset)
print("🌊 Стримы TinyStories готовы (train + validation)")


In [ ]:
# Загружаем best_model.pt (там шаг 20000)
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/checkpoint_step_100000.pt'
checkpoint = torch.load(path_best, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# если в best_model.pt сохранён scheduler, загрузите и его
if 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_step = checkpoint['step']  # должно быть 20000

# Затем продолжайте цикл с start_step+1 до 40000

In [ ]:
# второй цикл обучения
# ======================== ГИПЕРПАРАМЕТРЫ ========================
batch_size = 4
block_size = LeechConfig.block_size
learning_rate = 5e-5
total_steps = 100000
save_every = 1000
log_every = 100
gen_every = 1000

# ======================== ОПТИМИЗАТОР И ПЛАНИРОВЩИК ========================
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)  # Умеренный weight_decay

warmup_steps = 1000
total_steps = 100000   # или ваше общее число шагов
# После загрузки чекпоинта (start_step = load_latest_checkpoint(...))
if start_step >= 20000:  # если мы действительно перешли рубеж 20k
    print("🔄 Начинаем второй цикл обучения с LR=1e-4")
    # Сбрасываем learning rate в оптимизаторе
    for param_group in optimizer.param_groups:
        param_group['lr'] = 5e-5

    # Создаём новый планировщик на оставшиеся 20000 шагов
    warmup_steps = 1000
    remaining_steps = 20000  # можно вычислить как total_steps - start_step, но здесь мы фиксируем вторую половину
    warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=remaining_steps - warmup_steps, eta_min=5e-5)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])
else:
    # Если по какой-то причине start_step < 20000, оставляем старый планировщик (маловероятно)
    pass

In [ ]:
# второй цикл обучения
# ======================== ГИПЕРПАРАМЕТРЫ ========================
batch_size = 4
block_size = LeechConfig.block_size
learning_rate = 5e-5
total_steps = 100000
save_every = 1000
log_every = 100
gen_every = 1000

# ======================== ОПТИМИЗАТОР И ПЛАНИРОВЩИК ========================
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)  # Умеренный weight_decay

warmup_steps = 1000
total_steps = 100000   # или ваше общее число шагов

In [ ]:
#  начальное обучение до 20К
# ======================== ГИПЕРПАРАМЕТРЫ ========================
batch_size = 4
block_size = cfg.block_size
learning_rate = 5e-5
total_steps = 100000
save_every = 1000
log_every = 100
gen_every = 1000

# ======================== ОПТИМИЗАТОР И ПЛАНИРОВЩИК ========================
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)  # Умеренный weight_decay

warmup_steps = 1000
total_steps = 100000   # или ваше общее число шагов

# 1. Разогрев: от 0.01 * lr до lr
warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)

# 2. Косинусное затухание: от lr до eta_min
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps, eta_min=1e-5)

# Объединяем
scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])

# Загружаем чекпоинт (модель, оптимизатор и scheduler)
start_step = load_latest_checkpoint(model, optimizer, scheduler)   # модифицируем функцию

In [ ]:
# ======================== ЦИКЛ ОБУЧЕНИЯ ========================
print(f"\n🚀 Запуск обучения с шага {start_step} до {total_steps}")
model.train()

#for _ in range(5): next(train_iter)
#print("💎 Стрим прогрет. Запускаем Резонанс...")

best_val_loss = float('inf')

for step in range(start_step + 1, total_steps + 1):
    xb, yb = get_batch_streaming(train_iter, batch_size, block_size, device)
    if xb is None:
        train_iter = iter(train_dataset)
        continue

    logits, _, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()  # обновляем learning rate
    if step % log_every == 0:
        print(f"📊 Шаг {step:5d} | Train Loss: {loss.item():.4f}")

    if step % log_every == 0:
        model.eval()
        with torch.no_grad():
            xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is None:
                val_iter = iter(val_dataset)
                xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is not None:
                _, _, val_loss = model(xb_val, yb_val)
                print(f"         | Validation Loss: {val_loss.item():.4f}")
        model.train()

    if step % gen_every == 0:
        model.eval()
        with torch.no_grad():
            test_prompt = "who is Lily? "
            context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)
            print(f"\n🎭 Генерация (шаг {step}): ", end='')
            for _ in range(50):
                idx_cond = context[:, -block_size:]
                logits_gen, _ , _ = model(idx_cond)
                logits_gen = logits_gen[0, -1, :].clone()
                past_tokens = context[0, -20:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits_gen[t] -= (1.5 * count)
                last_token = context[0, -1].item()
                last_piece = sp.id_to_piece(last_token)
                if last_piece in ":,.!?;":
                    for p in ":,.!?;":
                        pid = sp.piece_to_id(p)
                        if pid != -1:
                            logits_gen[pid] -= 50.0
                probs = torch.softmax(logits_gen / 0.8, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                context = torch.cat((context, next_token.unsqueeze(0)), dim=1)
                piece = sp.id_to_piece(next_token.item())
                if piece == '<0x22>':
                    piece = '"'
                elif piece == '<0x0A>':
                    piece = '\n'
                elif piece.startswith('▁'):
                    piece = ' ' + piece[1:]
                elif piece.startswith('<0x') and piece.endswith('>'):
                    continue
                print(piece, end='', flush=True)
            print("\n" + "-"*60)
        model.train()

    if step % save_every == 0:
      save_checkpoint(step, model, optimizer, scheduler, loss.item())
      if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(step, model, optimizer, scheduler, val_loss.item(), is_best=True)   # step, а не total_steps

save_checkpoint(total_steps, model, optimizer, scheduler, loss.item())  # добавили scheduler
print("🏁 Обучение завершено!")

In [ ]:
def generate_with_resonance(model, prompt_ids, max_new_tokens=500, temperature=0.5, top_k=50):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([prompt_ids], device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, hidden, _ = model(input_ids, use_resonator=True)  # резонатор активен
            next_logits = logits[0, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits[next_logits < v[-1]] = -float('Inf')
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)
    return input_ids[0].tolist()

In [ ]:
# ======================== СОХРАНЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ ========================
print("\n💾 Сохранение модели Leech-GPT...")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': cfg,
    'leech_basis': model.leech_basis.cpu()
}, 'leech_model_final.pth')
print(f"✅ Модель сохранена в 'leech_model_final.pth'")

# ======================== ПРИМЕР ГЕНЕРАЦИИ  ========================
# Загружаем лучший чекпоинт (если нужно)
# path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_model.pt'
# checkpoint = torch.load(path_best, map_location=device, weights_only=False)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.to(device)
# model.eval()

In [ ]:
import re

# ======================== ПРИМЕР ГЕНЕРАЦИИ  ========================
# Загружаем лучший чекпоинт (если нужно)
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_model.pt'
checkpoint = torch.load(path_best, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

def clean_token(piece):
    """Преобразует специальные токены в читаемый вид или пропускает их."""
    if piece == '<0x22>':
        return '"'
    elif piece == '<0x0A>':
        return '\n'
    elif piece.startswith('▁'):
        return ' ' + piece[1:]  # пробел перед словом
    elif piece.startswith('<0x') and piece.endswith('>'):
        return None  # пропускаем байтовые токены
    elif piece == '<pad>':
        return None
    else:
        return piece

def generate_clean(model, prompt, max_new_tokens=80, temperature=0.5, top_k=50):
    model.eval()
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = context[:, -model.cfg.block_size:]
            logits, _, _ = model(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context, next_token], dim=1)

    tokens = context[0].tolist()
    # Очистка
    output = []
    for t in tokens:
        piece = sp.id_to_piece(t)
        cleaned = clean_token(piece)
        if cleaned is not None:
            output.append(cleaned)
    return ''.join(output).strip()

# Пример использования
test_prompt = "Once upon a time  "
print(generate_clean(model, test_prompt))

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict, deque
import time

def generate_long(
    model,
    prompt_ids,
    max_new_tokens=1500,
    temperature=0.5,
    top_k=50,
    top_p=0.95,
    repetition_penalty_start=1.5,
    repetition_penalty_max=2.5,
    penalty_increase_steps=1000,
    ngram_size=3,
    ngram_penalty=1.2,
    window_size=100,
    device=None,
    sp=None,     # sentencepiece processor (обязателен для вывода)
    stream_output=True,           # печатать по ходу генерации
    delay=0.0                     # задержка между токенами (сек)
):
    """
    Генерирует текст с расширенными параметрами борьбы с зацикливанием
    и опциональным потоковым выводом.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()
    input_ids = torch.tensor([prompt_ids], device=device)
    generated = input_ids[0].tolist()

    # Для n-граммного штрафа
    ngram_counts = defaultdict(int)
    recent_ngrams = deque(maxlen=window_size)

    # Печатаем начало (промпт) если нужно
    if stream_output and sp is not None:
        prompt_text = decode(prompt_ids)  # используем вашу функцию decode
        print(prompt_text, end='', flush=True)

    for step in range(max_new_tokens):
        # Динамический repetition penalty
        if step < penalty_increase_steps:
            rep_penalty = repetition_penalty_start
        else:
            progress = (step - penalty_increase_steps) / (max_new_tokens - penalty_increase_steps)
            rep_penalty = repetition_penalty_start + progress * (repetition_penalty_max - repetition_penalty_start)

        # Обрезаем до block_size
        idx_cond = input_ids[:, -model.cfg.block_size:]
        with torch.no_grad():
            logits, _, _ = model(idx_cond)
        logits = logits[0, -1, :].clone()

        # Штраф за повтор токенов
        if rep_penalty != 1.0:
            # Штрафуем токены, встреченные в последних window_size позициях
            for token in set(generated[-window_size:]):
                logits[token] /= rep_penalty

        # Штраф за повтор n-грамм (наказываем первый токен n-граммы, если она уже встречалась)
        if ngram_penalty != 1.0 and len(generated) >= ngram_size:
            last_ngram = tuple(generated[-ngram_size:])
            if last_ngram in ngram_counts and ngram_counts[last_ngram] > 0:
                logits[last_ngram[0]] /= ngram_penalty

        # Top-k фильтр
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[-1]] = -float('Inf')

        # Top-p (nucleus) фильтр
        if top_p is not None and top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            # Сдвигаем, чтобы оставить первый токен
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = -float('Inf')

        # Выбор следующего токена
        probs = F.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=device)], dim=1)
        generated.append(next_token)

        # Обновление статистики n-грамм
        if len(generated) >= ngram_size:
            ngram = tuple(generated[-ngram_size:])
            ngram_counts[ngram] += 1
            recent_ngrams.append(ngram)
            if len(recent_ngrams) == window_size:
                oldest = recent_ngrams[0]
                ngram_counts[oldest] -= 1
                if ngram_counts[oldest] == 0:
                    del ngram_counts[oldest]

        # Потоковый вывод
        if stream_output and sp is not None:
            piece = sp.id_to_piece(next_token)
            # Фильтрация и форматирование (как в вашем цикле обучения)
            if piece == '<0x22>':
                piece = '"'
            elif piece == '<0x0A>':
                piece = '\n'
            elif piece.startswith('▁'):
                piece = ' ' + piece[1:]
            elif piece.startswith('<0x') and piece.endswith('>'):
                continue          # пропускаем байтовые токены
            elif piece == '<pad>':
                continue
            print(piece, end='', flush=True)
            if delay > 0:
                time.sleep(delay)

    if stream_output:
        print()  # завершающая новая строка
    return input_ids[0].tolist()

# Пример вызова
test_prompt = "Once upon a time Lily and her dog Max "
encoded_prompt = encode(test_prompt)   # превращаем строку в токены

generated_tokens = generate_long(
    model,
    encoded_prompt,
    max_new_tokens=100,
    device=device,
    sp=sp,                # передаём токенайзер
    stream_output=True,
    #delay=0.02,           # небольшая задержка для читаемости
    repetition_penalty_start=1.5,
    repetition_penalty_max=2.5,
    ngram_penalty=1.2
)

# При необходимости можно получить полный текст
full_text = decode(generated_tokens)


In [ ]:


# ==================== ФУНКЦИЯ ГЕНЕРАЦИИ С МОНИТОРИНГОМ ====================
def leech_generate(model, start_tokens, max_len=100, temperature=0.2,
                   resonance_check=True, leech_basis=None, threshold=0.55):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([start_tokens], device=device)
    if resonance_check:
        decoder = DreamDecoder(leech_basis, threshold)

    print("--- LEECH GENERATION ---")
    with torch.no_grad():
        for step in range(max_len):
            logits, hidden, _ = model(input_ids)     # получаем hidden
            next_token_logits = logits[0, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)

            if resonance_check:
                last_hidden = hidden[0, -1, :]   # последнее скрытое состояние
                res, status = decoder.check(last_hidden)
               # print(f"Step {step:02d} | Resonance: {res:.6f} | Status: {status}")
    return input_ids

# Пример использования после обучения:
start = encode("Who are you?")
generated_ids = leech_generate(model, start, max_len=150, leech_basis=model.leech_basis)
print(decode(generated_ids[0].tolist()))

In [ ]:
from google.colab import files
files.download('leech_model_final.pth')

In [ ]:
# ========== ФАЗА 2: ДООБУЧЕНИЕ С РЕЗОНАТОРОМ ========================
print("🔄 Загрузка лучшей модели из первой фазы...")
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_model.pt'
checkpoint = torch.load(path_best, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Замораживаем все веса модели
for param in model.parameters():
    param.requires_grad = False

# Создаём резонатор и добавляем в модель
resonator = LeechResonanceBiasing(cfg.d_model, cfg.vocab_size, model.leech_basis, alpha_init=0.1).to(device)
resonator.compute_token_proj(model.tok_emb.weight.detach())
model.resonator = resonator  # сохраняем как атрибут

# Оптимизатор только для alpha
optimizer = torch.optim.AdamW([resonator.alpha], lr=1e-3, weight_decay=0.1)

# Планировщик для alpha (опционально, можно оставить постоянный lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-5)

# Гиперпараметры дообучения
batch_size = 4
block_size = cfg.block_size
resonator_steps = 10000          # количество шагов дообучения
log_every = 100
gen_every = 1000                 # генерация каждые 200 шагов
save_every = 1000                # сохранение чекпоинта

print("\n🚀 Дообучение с резонатором...")
model.train()
best_val_loss = float('inf')

for step in range(1, resonator_steps + 1):
    xb, yb = get_batch_streaming(train_iter, batch_size, block_size, device)
    if xb is None:
        train_iter = iter(train_dataset)
        continue

    # Важно: use_resonator=True
    logits, hidden, loss = model(xb, targets=yb, use_resonator=True)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([resonator.alpha], max_norm=1.0)
    optimizer.step()
    if scheduler:
        scheduler.step()

    # Логирование
    if step % log_every == 0:
        current_alpha = resonator.alpha.item()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"📊 Шаг {step:4d} | Loss: {loss.item():.4f} | Alpha: {current_alpha:.6f} | LR: {current_lr:.2e}")

    # Валидация
    if step % log_every == 0:
        model.eval()
        with torch.no_grad():
            xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is None:
                val_iter = iter(val_dataset)
                xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is not None:
                _, _, val_loss = model(xb_val, yb_val, use_resonator=True)
                print(f"         | Validation Loss: {val_loss.item():.4f}")
        model.train()

    # Генерация примера
    if step % gen_every == 0:
        model.eval()
        with torch.no_grad():
            test_prompt = "who is Lily? "
            context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)
            print(f"\n🎭 Генерация (шаг {step}): ", end='')
            for _ in range(50):
                idx_cond = context[:, -block_size:]
                logits_gen, _, _ = model(idx_cond, use_resonator=True)  # резонатор активен
                logits_gen = logits_gen[0, -1, :].clone()
                past_tokens = context[0, -20:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits_gen[t] -= (1.5 * count)
                last_token = context[0, -1].item()
                last_piece = sp.id_to_piece(last_token)
                if last_piece in ":,.!?;":
                    for p in ":,.!?;":
                        pid = sp.piece_to_id(p)
                        if pid != -1:
                            logits_gen[pid] -= 50.0
                probs = torch.softmax(logits_gen / 0.8, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                context = torch.cat((context, next_token.unsqueeze(0)), dim=1)
                piece = sp.id_to_piece(next_token.item())
                if piece == '<0x22>':
                    piece = '"'
                elif piece == '<0x0A>':
                    piece = '\n'
                elif piece.startswith('▁'):
                    piece = ' ' + piece[1:]
                elif piece.startswith('<0x') and piece.endswith('>'):
                    continue
                print(piece, end='', flush=True)
            print("\n" + "-"*60)
        model.train()

    # Сохранение чекпоинта
    if step % save_every == 0:
        checkpoint_path = f'/content/drive/MyDrive/leech_tinystories_checkpoints/resonator_step_{step}.pt'
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'resonator_state_dict': resonator.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
            'alpha': resonator.alpha.item(),
            'config': cfg,
        }, checkpoint_path)
        print(f"💾 Чекпоинт сохранён: {checkpoint_path}")

        # Сохраняем лучшую модель по валидационному loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_resonator_model.pt'
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'resonator_state_dict': resonator.state_dict(),
                'loss': val_loss.item(),
                'alpha': resonator.alpha.item(),
                'config': cfg,
            }, best_path)
            print(f"🏆 Новая лучшая модель с резонатором! Loss: {val_loss:.4f}")

# Финальное сохранение
final_path = '/content/drive/MyDrive/leech_tinystories_checkpoints/final_model_with_resonator.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'resonator_state_dict': resonator.state_dict(),
    'alpha': resonator.alpha.item(),
    'config': cfg,
}, final_path)
print(f"✅ Дообучение завершено. Модель сохранена в {final_path}")